# Student Assignment Score: Linear Regression

This notebook uses `student_clustering.csv` to predict a student's **Assignment_Score** from their **Study_Hours** and **Attendance**. It covers basic data cleaning, visual exploration, a train/test split, a linear regression model, and test-set evaluation.

`Student_ID` is an identifier, not a meaningful measurement, so it is not used as a model input.

## 1. Import libraries

Pandas handles the table, Matplotlib and Seaborn make charts, and scikit-learn provides the model and evaluation tools.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

sns.set_theme(style="whitegrid")
%matplotlib inline

## 2. Load and inspect the data

The path check supports running this notebook from `Demo/`, the `Section-3/` project folder, or the parent workspace folder.

In [ ]:
data_file = Path("../data/student_clustering.csv")
if not data_file.exists():
    data_file = Path("data/student_clustering.csv")
if not data_file.exists():
    data_file = Path("Section-3/data/student_clustering.csv")
if not data_file.exists():
    data_file = Path("../Section-3/data/student_clustering.csv")

if not data_file.exists():
    raise FileNotFoundError("Could not find data/student_clustering.csv. Check the notebook and data folder locations.")

students = pd.read_csv(data_file)
students.head()

In [ ]:
print("Rows and columns:", students.shape)
print("\nColumn types and non-null counts:")
students.info()
print("\nSummary statistics:")
students.describe()

## 3. Clean the data

We remove duplicate records, convert the two inputs and target to numbers, and remove rows where one of those required values is missing. This dataset is expected to be clean already; the checks make the notebook safer if the CSV is updated.

In [ ]:
features = ["Study_Hours", "Attendance"]
target = "Assignment_Score"
required_columns = features + [target]

print("Missing values before cleaning:")
print(students[required_columns].isna().sum())
print("Duplicate rows before cleaning:", students.duplicated().sum())

students = students.drop_duplicates().copy()
for column in required_columns:
    students[column] = pd.to_numeric(students[column], errors="coerce")
students = students.dropna(subset=required_columns).reset_index(drop=True)

print("\nRows after cleaning:", len(students))
print("Missing values after cleaning:")
print(students[required_columns].isna().sum())

## 4. Explore and visualize

First, view the distributions of the numeric columns. Then compare each input with the assignment score. These plots help us see patterns, but they do not prove that one variable causes another.

In [ ]:
students[required_columns].hist(figsize=(10, 7), bins=15, edgecolor="black")
plt.suptitle("Student data distributions")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for axis, feature in zip(axes, features):
    sns.regplot(data=students, x=feature, y=target, ax=axis,
                scatter_kws={"alpha": 0.6}, line_kws={"color": "red"})
    axis.set_title(f"{feature} vs {target}")
plt.tight_layout()
plt.show()

In [ ]:
correlations = students[required_columns].corr()
sns.heatmap(correlations, annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation between numeric columns")
plt.tight_layout()
plt.show()

## 5. Split into training and test data

The model learns from 80% of the rows. The remaining 20% are held aside for a fair check on data it did not train on. A fixed `random_state` makes the split repeatable.

In [ ]:
X = students[features]
y = students[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

## 6. Train the linear regression model

The model estimates a straight-line relationship using both inputs:

`Predicted Assignment_Score = intercept + (Study_Hours coefficient × Study_Hours) + (Attendance coefficient × Attendance)`

In [ ]:
model = LinearRegression()
model.fit(X_train, y_train)

print(f"Intercept: {model.intercept_:.2f}")
for feature, coefficient in zip(features, model.coef_):
    print(f"{feature} coefficient: {coefficient:.2f}")

Each coefficient describes the predicted score change for a one-unit increase in that input, **while holding the other input constant**. These are associations learned from this dataset, not proof of cause and effect.

## 7. Evaluate on the test data

- **MAE**: average absolute prediction error, in score points; lower is better.
- **MSE**: average squared error; larger mistakes count more heavily; lower is better.
- **RMSE**: square root of MSE, also in score points; lower is better.
- **R²**: proportion of test-score variation explained by the model; closer to 1 is generally better. It can be negative if the model performs worse than predicting the test-set mean.

In [ ]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f} score points")
print(f"MSE:  {mse:.2f}")
print(f"RMSE: {rmse:.2f} score points")
print(f"R²:   {r2:.3f}")

## 8. Check predictions and residuals

In the first plot, points close to the dashed line are more accurate predictions. In the residual plot, residual means actual score minus predicted score; a useful model should show residuals scattered around zero without a strong pattern.

In [ ]:
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].scatter(y_test, y_pred, alpha=0.7)
score_min = min(y_test.min(), y_pred.min())
score_max = max(y_test.max(), y_pred.max())
axes[0].plot([score_min, score_max], [score_min, score_max], "r--")
axes[0].set_xlabel("Actual assignment score")
axes[0].set_ylabel("Predicted assignment score")
axes[0].set_title("Actual vs predicted")

axes[1].scatter(y_pred, residuals, alpha=0.7)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted assignment score")
axes[1].set_ylabel("Residual (actual - predicted)")
axes[1].set_title("Residual plot")

plt.tight_layout()
plt.show()

## 9. Conclusion

Use the test-set metrics and plots above to describe how well this model predicts assignment scores. Mention the typical error in score points (MAE or RMSE), whether the R² is useful, and any visible residual pattern. Results describe this dataset and may not generalize to other students.